# ASL Model Evaluation (Processed Test Split)

This notebook loads a saved Keras model from `models/` and evaluates it on `data/asl_processed/test`.

- Metrics: accuracy + loss
- Classification report
- Confusion matrix
- Example predictions / misclassifications


In [1]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data/asl_processed")
MODEL_DIR = Path("../models/asl_model")

meta = json.loads((MODEL_DIR / "meta.json").read_text())
class_names = json.loads((MODEL_DIR / "class_names.json").read_text())
image_size = int(meta["image_size"]) 

_bundle = MODEL_DIR / "model.keras"
model = tf.keras.models.load_model(_bundle if _bundle.is_file() else MODEL_DIR)
model.summary()

2026-05-01 22:43:10.337650: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-05-01 22:43:10.337815: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-01 22:43:10.337819: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1777689790.338134 2347729 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1777689790.338417 2347729 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ backbone (Functional)           │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 36)             │        46,116 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,187,921 (15.98 MB)

 Trainable params: 46,116 (180.14 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 92,234 (360.29 KB)

In [2]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    labels="inferred",
    label_mode="int",
    image_size=(image_size, image_size),
    batch_size=64,
    shuffle=False,
)

test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

results = model.evaluate(test_ds, verbose=1)
dict(zip(model.metrics_names, results))

Found 7200 files belonging to 36 classes.


2026-05-01 22:43:18.612883: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


113/113 ━━━━━━━━━━━━━━━━━━━━ 47s 400ms/step - acc: 0.9968 - loss: 0.1732


{'loss': 0.17321959137916565, 'compile_metrics': 0.9968055486679077}

In [ ]:
from datetime import datetime

y_true = []
y_prob = []

for xb, yb in test_ds:
    y_true.append(yb.numpy())
    y_prob.append(model.predict(xb, verbose=0))

y_true = np.concatenate(y_true)
y_prob = np.concatenate(y_prob)
y_pred = y_prob.argmax(axis=1)

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

_root = Path.cwd().resolve()
reports_dir = _root / "reports" if _root.name == "notebooks" else _root / "notebooks" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
modelname = MODEL_DIR.name
stamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
report_path = reports_dir / f"{modelname}_{stamp}.txt"
report_path.write_text(report)
print(f"Saved classification report to: {report_path}")


              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       200
           1     1.0000    1.0000    1.0000       200
           2     1.0000    0.9950    0.9975       200
           3     1.0000    1.0000    1.0000       200
           4     1.0000    0.9900    0.9950       200
           5     1.0000    1.0000    1.0000       200
           6     0.9747    0.9650    0.9698       200
           7     0.9800    0.9800    0.9800       200
           8     0.9800    0.9800    0.9800       200
           9     1.0000    1.0000    1.0000       200
           A     1.0000    0.9950    0.9975       200
           B     1.0000    1.0000    1.0000       200
           C     1.0000    1.0000    1.0000       200
           D     1.0000    1.0000    1.0000       200
           E     1.0000    1.0000    1.0000       200
           F     1.0000    1.0000    1.0000       200
           G     1.0000    1.0000    1.0000       200
           H     1.0000    

2026-05-01 22:50:38.392391: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues", cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Pred")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
inv = {i: name for i, name in enumerate(class_names)}

# show a few misclassifications
mis = np.where(y_true != y_pred)[0]
print("Misclassified:", len(mis), "/", len(y_true))

if len(mis) > 0:
    idxs = mis[:16]
    # recreate a non-batched dataset to fetch the corresponding images
    raw_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / "test",
        labels="inferred",
        label_mode="int",
        image_size=(image_size, image_size),
        batch_size=1,
        shuffle=False,
    )

    imgs = []
    labs = []
    for xb, yb in raw_ds.take(int(idxs[-1]) + 1):
        imgs.append(xb[0].numpy().astype(np.uint8))
        labs.append(int(yb[0].numpy()))

    plt.figure(figsize=(12, 12))
    for i, k in enumerate(idxs):
        plt.subplot(4, 4, i + 1)
        plt.imshow(imgs[int(k)])
        t = inv[int(y_true[int(k)])]
        p = inv[int(y_pred[int(k)])]
        plt.title(f"T:{t}  P:{p}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()